# 02 — Aggregate · Annotate · Normalize · FeatureSelect (MERGED)

> Versão • 2026-04-23 13:25

Notebook **unificado**: permite construir o dataset a partir de **SQLite** (Per_* tables) **ou** de **CSVs por compartimento** (Cells/Cytoplasm/Nuclei).

Principais pontos:
- Preserva e padroniza **metadados** (`Metadata_Plate`, `Metadata_Well`, `Metadata_Site`).
- Prefixa *features* por compartimento.
- Agrega por **poço** (mediana).
- Anota por **platemap** (index flexível).
- Normaliza *plate-wise* (`mad_robustize`) com `pycytominer` (modo `all` ou `negcon`).
- Seleção de *features* com `FeatureSelect`.


## Passo 1 — Configuração do experimento

In [13]:
from __future__ import annotations

# Standard library
import re
import sqlite3
import string
from pathlib import Path
from typing import Optional

# Third-party
import numpy as np
import pandas as pd
from pycytominer import normalize as pc_normalize
from pycytominer.cyto_utils import infer_cp_features

# Pycytominer feature_select (API funcional preferível; fallback para versões antigas)
try:
    from pycytominer import feature_select as pc_feature_select  # functional API

    _FEATURESELECT_MODE = "functional"
except Exception:
    from pycytominer.operations import feature_select as _fs_mod  # legacy API

    PCFeatureSelect = _fs_mod.FeatureSelect
    _FEATURESELECT_MODE = "class"

# Display settings
pd.set_option("display.max_columns", 200)

## Passo 2 — Funções auxiliares

In [14]:
def find_repo_root(
    start: Path, markers=(".git", "pyproject.toml", "pixi.toml")
) -> Path:
    start = start.resolve()
    for p in (start, *start.parents):
        if any((p / m).exists() for m in markers):
            return p
    # fallback: sobe até 5 níveis (evita travar), mas melhor ter marker no repo
    return start.parents[5] if len(start.parents) > 5 else start


# Em VS Code, normalmente cwd == pasta do notebook
NOTEBOOK_DIR = Path.cwd().resolve()

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)


In [15]:
# =========================================================
# Contexto do notebook / experimento
# =========================================================
NOTEBOOK_DIR = Path.cwd().resolve()
EXPERIMENT_ID = NOTEBOOK_DIR.name  # ex.: "7_dias"

# =========================================================
# Pastas “canônicas” do repositório
# (todas as outras rotas derivam daqui)
# =========================================================
WORKSPACE_DIR = REPO_ROOT / "workspace"

# =========================================================
# Pastas por área do workflow (sempre por EXPERIMENT_ID)
# =========================================================
METADATA_DIR = WORKSPACE_DIR / "metadata" / EXPERIMENT_ID
ANALYSIS_DIR = WORKSPACE_DIR / "analysis" / EXPERIMENT_ID
PROFILES_DIR = WORKSPACE_DIR / "profiles" / EXPERIMENT_ID

# =========================================================
# Subpastas padrão de outputs / figures
# (padroniza para não misturar arquivos na raiz do experimento)
# =========================================================
ANALYSIS_OUT_DIR = ANALYSIS_DIR / "outputs"
PROFILES_OUT_DIR = PROFILES_DIR / "outputs"
FIGS_DIR = PROFILES_DIR / "figs"
CACHE_DIR = PROFILES_OUT_DIR / "cache"

# =========================================================
# Arquivos de metadata (sempre dentro de METADATA_DIR)
# =========================================================
BARCODE_PLATEMAP_CSV = METADATA_DIR / "barcode_platemap.csv"
PLATEMAP_DIR = METADATA_DIR / "platemap"

# =========================================================
# Inputs do Jupyter 2
# (saída do Jupyter 1 / etapa anterior)
# =========================================================
INPUT_CSV = ANALYSIS_OUT_DIR / "single_cell_profiles.csv"

# =========================================================
# Checkpoints / caches do Jupyter 2
# (artefatos intermediários desta etapa)
# =========================================================
SINGLE_CELL_LOADED_PARQUET = CACHE_DIR / "single_cell_loaded.parquet"
SINGLE_CELL_ANNOTATED_PARQUET = CACHE_DIR / "single_cell_annotated.parquet"
SINGLE_CELL_READY_PARQUET = CACHE_DIR / "single_cell_ready.parquet"
PER_WELL_AGGREGATED_PARQUET = CACHE_DIR / "per_well_aggregated.parquet"
PER_WELL_NORMALIZED_PARQUET = CACHE_DIR / "per_well_normalized.parquet"

# =========================================================
# Outputs do Jupyter 2
# (artefatos gerados nesta etapa)
# =========================================================
PER_WELL_FEATURES_SELECTED_PARQUET = (
    PROFILES_OUT_DIR / "per_well_features_selected.parquet"
)

# =========================================================
# Garantir que pastas de saída existem
# =========================================================
for d in [ANALYSIS_OUT_DIR, PROFILES_OUT_DIR, FIGS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================================================
# Checagens rápidas (fail fast)
# =========================================================
assert METADATA_DIR.exists(), f"METADATA_DIR não existe: {METADATA_DIR}"
assert BARCODE_PLATEMAP_CSV.exists(), f"Não achei: {BARCODE_PLATEMAP_CSV}"
assert PLATEMAP_DIR.exists(), f"PLATEMAP_DIR não existe: {PLATEMAP_DIR}"
assert INPUT_CSV.exists(), f"Não achei INPUT_CSV: {INPUT_CSV}"

In [16]:
def _status(exists: bool | None) -> str:
    return "—" if exists is None else ("✅ existe" if exists else "❌ não existe")


def row(label: str, value, exists: bool | None = None) -> None:
    # value pode ser Path ou str
    if isinstance(value, Path):
        v = str(value)
        e = value.exists() if exists is None else exists
    else:
        v = str(value)
        e = exists  # para strings, geralmente None

    print(f"{label:<20} {_status(e):<12} {v}")


# Agora imprime tudo em 1 linha, incluindo EXPERIMENT_ID
row("EXPERIMENT_ID", EXPERIMENT_ID)
row("NOTEBOOK_DIR", NOTEBOOK_DIR)
row("REPO_ROOT", REPO_ROOT)
row("METADATA_DIR", METADATA_DIR)
row("BARCODE_PLATE", BARCODE_PLATEMAP_CSV)
row("PLATEMAP_DIR", PLATEMAP_DIR)
row("INPUT_CSV", INPUT_CSV)
row("OUTPUT_PARQUET:", PER_WELL_FEATURES_SELECTED_PARQUET)

EXPERIMENT_ID        —            2026_04_Huh7_NPPS_24h
NOTEBOOK_DIR         ✅ existe     /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/analysis/2026_04_Huh7_NPPS_24h
REPO_ROOT            ✅ existe     /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7
METADATA_DIR         ✅ existe     /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/metadata/2026_04_Huh7_NPPS_24h
BARCODE_PLATE        ✅ existe     /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/metadata/2026_04_Huh7_NPPS_24h/barcode_platemap.csv
PLATEMAP_DIR         ✅ existe     /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/metadata/2026_04_Huh7_NPPS_24h/platemap
INPUT_CSV            ✅ existe     /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/analysis/2026_04_Huh7_NPPS_24h/outputs/single_cell_profi

In [17]:
def _norm_well(x: str) -> str:
    """Normaliza poços para o formato A01–H12."""
    s = str(x).strip().upper()
    import re as _re

    m = _re.match(r"^([A-H])0?([1-9]|1[0-2])$", s)
    return f"{m.group(1)}{int(m.group(2)):02d}" if m else s


def ensure_core_metadata(
    df: pd.DataFrame, plate_value: Optional[str] = None
) -> pd.DataFrame:
    """
    Padroniza colunas de metadados sem remover nada.
    - Renomeia variações comuns -> Metadata_Plate/Well/Site
    - Normaliza Metadata_Well (A01–H12)
    """
    rename_map = {
        "Image_Metadata_Plate": "Metadata_Plate",
        "Image_Metadata_Well": "Metadata_Well",
        "Image_Metadata_Site": "Metadata_Site",
        "Site": "Metadata_Site",
        # variações ocasionais
        "Metadata_WellID": "Metadata_Well",
        "Well": "Metadata_Well",
        "Plate": "Metadata_Plate",
    }
    cols_to_rename = {k: v for k, v in rename_map.items() if k in df.columns}
    if cols_to_rename:
        df = df.rename(columns=cols_to_rename)
    if "Metadata_Plate" not in df.columns and plate_value is not None:
        df["Metadata_Plate"] = plate_value
    if "Metadata_Well" in df.columns:
        df["Metadata_Well"] = df["Metadata_Well"].map(_norm_well)
    return df


def dedupe_meta(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicatas e faz 'coalesce' de Metadata_* (ex.: _x/_y), mantendo meta no início."""
    META_KEYS = ["Metadata_Plate", "Metadata_Well", "Metadata_Site"]
    df = df.loc[:, ~df.columns.duplicated()].copy()
    for k in META_KEYS:
        cols = [c for c in df.columns if c == k or c.startswith(k + "_")]
        if len(cols) > 1:
            base = cols[0]
            for extra in cols[1:]:
                df[base] = df[base].where(df[base].notna(), df[extra])
            for extra in cols[1:]:
                if extra in df.columns:
                    df.drop(columns=extra, inplace=True)
    meta_first = [c for c in META_KEYS if c in df.columns]
    return df[meta_first + [c for c in df.columns if c not in meta_first]]

## Passo 3 — Carregar ou retomar análise

In [ ]:
# Carrega artefatos auxiliares, se existirem
if PER_WELL_AGGREGATED_PARQUET.exists():
    df_agg = pd.read_parquet(PER_WELL_AGGREGATED_PARQUET)
    print("Também carregado df_agg:", df_agg.shape)

if PER_WELL_NORMALIZED_PARQUET.exists():
    df_norm = pd.read_parquet(PER_WELL_NORMALIZED_PARQUET)
    print("Também carregado df_norm:", df_norm.shape)

if PER_WELL_FEATURES_SELECTED_PARQUET.exists():
    df_fs = pd.read_parquet(PER_WELL_FEATURES_SELECTED_PARQUET)
    print("Também carregado df_fs:", df_fs.shape)

In [20]:
# =========================================================
# Retomar execução a partir do checkpoint mais avançado
# =========================================================
# Rode esta célula depois de:
# 1) imports
# 2) paths/configuração do experimento
# 3) funções auxiliares básicas
#
# Depois, continue a partir do passo indicado no print.
# =========================================================

if PER_WELL_FEATURES_SELECTED_PARQUET.exists():
    df_fs = pd.read_parquet(PER_WELL_FEATURES_SELECTED_PARQUET)
    print("✅ Checkpoint encontrado: Passo 8 — feature selection final")
    print("Retomando de df_fs:", df_fs.shape)
    print("➡️ Continue no Passo 9 — salvar/verificar artefatos finais.")

elif PER_WELL_NORMALIZED_PARQUET.exists():
    df_norm = pd.read_parquet(PER_WELL_NORMALIZED_PARQUET)
    print("✅ Checkpoint encontrado: Passo 7 — per-well normalizado")
    print("Retomando de df_norm:", df_norm.shape)
    print("➡️ Continue no Passo 8 — feature selection.")

elif PER_WELL_AGGREGATED_PARQUET.exists():
    df_agg = pd.read_parquet(PER_WELL_AGGREGATED_PARQUET)
    print("✅ Checkpoint encontrado: Passo 6 — per-well agregado")
    print("Retomando de df_agg:", df_agg.shape)
    print("➡️ Continue no Passo 7 — normalização.")

elif SINGLE_CELL_READY_PARQUET.exists():
    df_annot_ready = pd.read_parquet(SINGLE_CELL_READY_PARQUET)
    print("✅ Checkpoint encontrado: Passo 5 — single-cell anotado e limpo")
    print("Retomando de df_annot_ready:", df_annot_ready.shape)
    print("➡️ Continue no Passo 6 — agregação por poço.")

elif SINGLE_CELL_ANNOTATED_PARQUET.exists():
    df_annot = pd.read_parquet(SINGLE_CELL_ANNOTATED_PARQUET)
    print("✅ Checkpoint encontrado: Passo 4 — single-cell anotado")
    print("Retomando de df_annot:", df_annot.shape)
    print("➡️ Continue no Passo 5 — QC/limpeza antes da agregação.")

elif SINGLE_CELL_LOADED_PARQUET.exists():
    df = pd.read_parquet(SINGLE_CELL_LOADED_PARQUET)
    print("✅ Checkpoint encontrado: Passo 3 — single-cell carregado")
    print("Retomando de df:", df.shape)
    print("➡️ Continue no Passo 4 — anotação com platemap.")

else:
    df = pd.read_csv(INPUT_CSV, low_memory=False)
    df = ensure_core_metadata(df)
    df = dedupe_meta(df)
    df.to_parquet(SINGLE_CELL_LOADED_PARQUET, index=False)

    print("⚠️ Nenhum checkpoint encontrado.")
    print("CSV carregado e checkpoint criado.")
    print("df:", df.shape)
    print("➡️ Continue no Passo 4 — anotação com platemap.")

✅ Checkpoint encontrado: Passo 3 — single-cell carregado
Retomando de df: (113108, 1811)
➡️ Continue no Passo 4 — anotação com platemap.


## Passo 4 — Anotar com platemap
Seu barcode_platemap. csv deve mapear placa → arquivo de layout (ex.: layout _day1. csv). A função abaixo normaliza para colunas padrão: Metadata_Plate, filename, plate_map_name.

In [21]:
def read_barcode_platemap(index_csv: Path) -> pd.DataFrame:
    idx = pd.read_csv(index_csv)
    # mapeia nomes comuns -> padrão
    rename_map = {
        "Assay_Plate_Barcode": "Metadata_Plate",
        "Plate_Map_Name": "filename",
    }
    for k, v in rename_map.items():
        if k in idx.columns and v not in idx.columns:
            idx = idx.rename(columns={k: v})

    # se veio com outros nomes, tente deduzir
    if "Metadata_Plate" not in idx.columns:
        # tenta colunas candidatas
        for cand in ["Plate_Barcode", "Plate", "Barcode"]:
            if cand in idx.columns:
                idx = idx.rename(columns={cand: "Metadata_Plate"})
                break
    if "filename" not in idx.columns:
        for cand in ["Layout_File", "Platemap", "Plate_Map", "Plate_Map_Name"]:
            if cand in idx.columns:
                idx = idx.rename(columns={cand: "filename"})
                break

    # cria plate_map_name (nome sem .csv)
    if "plate_map_name" not in idx.columns and "filename" in idx.columns:
        idx["plate_map_name"] = (
            idx["filename"].astype(str).str.replace(".csv", "", regex=False)
        )

    # checagem mínima
    assert {"Metadata_Plate", "filename"}.issubset(
        idx.columns
    ), "barcode_platemap.csv precisa ter: Metadata_Plate, filename (e opcional plate_map_name)."

    # limpeza leve
    idx["filename"] = idx["filename"].astype(str).str.strip()
    idx["Metadata_Plate"] = idx["Metadata_Plate"].astype(str).str.strip()
    return idx


idx = read_barcode_platemap(BARCODE_PLATEMAP_CSV)
display(idx.head())

# Sanity-check: os arquivos existem na pasta metadado/platemap?
for fn in idx["filename"].unique():
    p = PLATEMAP_DIR / fn
    print(f"{fn}: existe? {p.exists()}  -> {p}")

,Metadata_Plate,filename,plate_map_name
0,260221_171841_Plate_1_R1,layout_day1.csv,layout_day1
1,260223_170756_Plate_1_R2,layout_day2.csv,layout_day2
2,260228_135225_Plate_1_R3,layout_day3.csv,layout_day3


layout_day1.csv: existe? True  -> /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/metadata/2026_04_Huh7_NPPS_24h/platemap/layout_day1.csv
layout_day2.csv: existe? True  -> /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/metadata/2026_04_Huh7_NPPS_24h/platemap/layout_day2.csv
layout_day3.csv: existe? True  -> /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/metadata/2026_04_Huh7_NPPS_24h/platemap/layout_day3.csv


### 4.1 Ler cada layout de metadado/platemap/ e anotar por placa
Detectamos a coluna que contém poços (ex.: Well, Metadata_Well, well_id…), renomeamos para Metadata_Well, normalizamos para A01–H12 e fazemos merge com df por Metadata_Well (por placa).

In [22]:
def _read_platemap_csv(csv_path: Path) -> pd.DataFrame:
    """
    Lê um layout padronizado com colunas:
    [plate_map_name, well_position, Cell_Type, Treatment, Concentration, Control_Type]

    - Renomeia well_position -> Metadata_Well (normalizado A01–H12)
    - Mantém as colunas originais
    - Cria colunas auxiliares Metadata_* usadas em etapas posteriores (opcional, mas útil)
    """
    pm = pd.read_csv(csv_path)

    required = [
        "plate_map_name",
        "well_position",
        "Cell_Type",
        "Treatment",
        "Concentration",
        "Control_Type",
    ]
    missing = [c for c in required if c not in pm.columns]
    if missing:
        raise KeyError(
            f"{csv_path.name} não segue o padrão. Faltam colunas: {missing}. "
            f"Esperado: {required}"
        )

    # poço padronizado para A01–H12
    pm = pm.rename(columns={"well_position": "Metadata_Well"})
    pm["Metadata_Well"] = pm["Metadata_Well"].astype(str).map(_norm_well)

    # cria aliases Metadata_* que muitas rotinas usam
    pm["Metadata_Control_Type"] = pm["Control_Type"].astype(str).str.strip().str.lower()
    pm["Metadata_Treatment"] = pm["Treatment"]
    pm["Metadata_Concentration"] = pm["Concentration"]
    pm["Metadata_Cell_Type"] = pm["Cell_Type"]
    
    # limpeza leve
    pm["plate_map_name"] = pm["plate_map_name"].astype(str).str.strip()

    return pm


def annotate_per_plate(
    df_profiles: pd.DataFrame, idx: pd.DataFrame, pm_dir: Path
) -> pd.DataFrame:
    """
    Junta perfis (por placa) com o layout correspondente.
    O índice (barcode_platemap.csv) deve ter colunas: Metadata_Plate, filename, plate_map_name
    E cada layout_day*.csv deve seguir o padrão fixo acima.
    """
    outs = []
    for plate, sub in df_profiles.groupby("Metadata_Plate", sort=False):
        row = idx.loc[idx["Metadata_Plate"] == plate]
        if row.empty:
            print(
                f"⚠️ {plate}: placa não listada em barcode_platemap.csv — mantendo sem anotação"
            )
            outs.append(sub)
            continue

        pm_file = row["filename"].iloc[0]
        pm_name_from_index = (
            row["plate_map_name"].iloc[0] if "plate_map_name" in row.columns else None
        )

        pm_path = pm_dir / pm_file
        if not pm_path.exists():
            print(
                f"⚠️ {plate}: layout não encontrado: {pm_path} — mantendo sem anotação"
            )
            outs.append(sub)
            continue

        pm = _read_platemap_csv(pm_path)

        # Se o index define um nome-canônico da placa, sobrescreve para consistência
        if pm_name_from_index is not None:
            pm["plate_map_name"] = pm_name_from_index

        # merge por poço (padronizado)
        merged = sub.merge(pm, on="Metadata_Well", how="left")
        outs.append(merged)
        print(f"✓ {plate}: anotado via {pm_file} — shape {merged.shape}")

    return pd.concat(outs, axis=0, ignore_index=True)

### 4.2 Harmonizar nomes de colunas de anotação → Metadata_*

Se o seu layout contém Control_Type, Treatment, Concentration, etc., criamos colunas espelho Metadata_* para facilitar normalização e filtros.

In [23]:
df_annot = annotate_per_plate(df, idx, PLATEMAP_DIR)
df_annot = dedupe_meta(df_annot)

display(
    df_annot[
        [
            c
            for c in [
                "Metadata_Plate",
                "Metadata_Well",
                "plate_map_name",
                "Control_Type",
                "Metadata_Control_Type",
            ]
            if c in df_annot.columns
        ]
    ].head()
)

✓ 260221_171841_Plate_1_R1: anotado via layout_day1.csv — shape (33841, 1820)
✓ 260223_170756_Plate_1_R2: anotado via layout_day2.csv — shape (35309, 1820)
✓ 260228_135225_Plate_1_R3: anotado via layout_day3.csv — shape (37459, 1820)


,Metadata_Plate,Metadata_Well,plate_map_name,Control_Type,Metadata_Control_Type
0,260221_171841_Plate_1_R1,B10,layout_day1,negcon,negcon
1,260221_171841_Plate_1_R1,B10,layout_day1,negcon,negcon
2,260221_171841_Plate_1_R1,B10,layout_day1,negcon,negcon
3,260221_171841_Plate_1_R1,B10,layout_day1,negcon,negcon
4,260221_171841_Plate_1_R1,B10,layout_day1,negcon,negcon


In [24]:
df_annot.to_parquet(SINGLE_CELL_ANNOTATED_PARQUET, index=False)
print("Checkpoint salvo:", SINGLE_CELL_ANNOTATED_PARQUET)

Checkpoint salvo: /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/cache/single_cell_annotated.parquet


> **Nota de limpeza do fluxo**  
> As células que tentavam reconstruir `df_annot` a partir de `df`, `df_agg` ou de aliases recriados manualmente foram removidas.  
> A partir daqui, o notebook assume **um único caminho**: carregar `df` → anotar com `annotate_per_plate()` → validar/limpar → agregar.


In [25]:
def compare_layout_vs_data(df_annot: pd.DataFrame, idx: pd.DataFrame, platemap_dir: Path):
    reports = {}
    for plate in sorted(df_annot["Metadata_Plate"].dropna().astype(str).unique()):
        row = idx.loc[idx["Metadata_Plate"].astype(str) == plate]
        if row.empty:
            reports[plate] = pd.DataFrame(
                {"info": [f"{plate}: não está em barcode_platemap.csv"]}
            )
            continue

        layout_file = platemap_dir / row["filename"].iloc[0]
        if not layout_file.exists():
            reports[plate] = pd.DataFrame(
                {"info": [f"{plate}: layout não encontrado: {layout_file.name}"]}
            )
            continue

        pm = _read_platemap_csv(layout_file)
        wells_layout = set(pm["Metadata_Well"].astype(str).map(_norm_well))
        wells_data = set(
            df_annot.loc[df_annot["Metadata_Plate"].astype(str) == plate, "Metadata_Well"]
            .astype(str)
            .map(_norm_well)
        )
        missing_in_layout = sorted(wells_data - wells_layout)
        reports[plate] = pd.DataFrame({"Metadata_Well": missing_in_layout})
    return reports


layout_gaps = compare_layout_vs_data(df_annot, idx, PLATEMAP_DIR)
for plate, rep in layout_gaps.items():
    print(f"\n== {plate} ==")
    display(rep.head(20))
    if "Metadata_Well" in rep.columns:
        print(f"Total faltando no layout: {len(rep)}")



== 260221_171841_Plate_1_R1 ==


,Metadata_Well


Total faltando no layout: 0

== 260223_170756_Plate_1_R2 ==


,Metadata_Well


Total faltando no layout: 0

== 260228_135225_Plate_1_R3 ==


,Metadata_Well


Total faltando no layout: 0


## Passo 5 — QC e limpeza antes da agregação

Primeiro removemos poços que aparecem nos dados, mas **não existem no layout**.  
Depois, opcionalmente, removemos **poços técnicos específicos** definidos manualmente.  
O resultado dessa etapa será `df_annot_ready`, que é a base usada para a agregação.


In [26]:
# 1) montar um DF com todos os (plate, well) faltantes
missing_pairs = []
for plate, rep in layout_gaps.items():
    if isinstance(rep, pd.DataFrame) and "Metadata_Well" in rep.columns:
        for w in rep["Metadata_Well"].dropna().astype(str):
            missing_pairs.append((plate, _norm_well(w)))

missing_df = pd.DataFrame(missing_pairs, columns=["Metadata_Plate", "Metadata_Well"])
if not missing_df.empty:
    missing_df["Metadata_Well"] = missing_df["Metadata_Well"].map(_norm_well)

# 2) normalizar wells no df_annot e fazer anti-join (remover)
df_tmp = df_annot.copy()
df_tmp["__well_norm__"] = df_tmp["Metadata_Well"].astype(str).map(_norm_well)

if missing_df.empty:
    to_drop = pd.Series(False, index=df_tmp.index)
else:
    to_drop = (
        df_tmp[["Metadata_Plate", "__well_norm__"]]
        .merge(
            missing_df.rename(columns={"Metadata_Well": "__well_norm__"}).drop_duplicates(),
            how="left",
            on=["Metadata_Plate", "__well_norm__"],
            indicator=True,
        )["_merge"]
        .eq("both")
    )

n_removed = int(to_drop.sum())
print(f"🧹 Removendo {n_removed} linhas que não existem no layout.")

df_annot_clean = df_tmp.loc[~to_drop].drop(columns="__well_norm__").copy()
print("df_annot_clean shape:", df_annot_clean.shape)


🧹 Removendo 0 linhas que não existem no layout.
df_annot_clean shape: (106609, 1820)


In [27]:
rem_counts = (
    df_tmp.loc[to_drop, ["Metadata_Plate"]].value_counts().rename("removed").to_frame()
)
print(rem_counts.head(20) if not rem_counts.empty else "Nenhuma linha removida por ausência no layout.")


Nenhuma linha removida por ausência no layout.


In [28]:
import re

# helper: normaliza A1 -> A01
_well_re = re.compile(r"^\s*([A-Za-z])\s*0*([0-9]{1,2})\s*$")


def norm_well(s):
    s = str(s).strip()
    m = _well_re.match(s)
    if m:
        return f"{m.group(1).upper()}{int(m.group(2)):02d}"
    if len(s) >= 2 and s[0].isalpha() and s[1:].isdigit():
        return s[0].upper() + s[1:].zfill(2)
    return s


# poços a excluir (ajuste conforme necessário)
EXCLUDE_WELLS = {
    "250704_104743_Plate_1": ["B02"],
    "250709_095049_Plate_1": [
        "B03",
        "B11",
        "C03",
        "C11",
        "D03",
        "D11",
        "E03",
        "E11",
        "F03",
        "F11",
        "G11",
    ],
}

df_annot_ready = df_annot_clean.copy()
df_annot_ready["__well_norm__"] = df_annot_ready["Metadata_Well"].map(norm_well)

rem_mask = pd.Series(False, index=df_annot_ready.index)
for plate, wells in EXCLUDE_WELLS.items():
    wells = [norm_well(w) for w in wells]
    rem_mask |= (df_annot_ready["Metadata_Plate"] == plate) & (
        df_annot_ready["__well_norm__"].isin(wells)
    )

n_removed = int(rem_mask.sum())
if n_removed:
    print(f"ℹ️ Removendo {n_removed} linhas de poços técnicos antes da agregação.")
    df_annot_ready = df_annot_ready.loc[~rem_mask].drop(columns="__well_norm__")
else:
    df_annot_ready = df_annot_ready.drop(columns="__well_norm__", errors="ignore")

print("df_annot_ready:", df_annot_ready.shape)


df_annot_ready: (106609, 1820)


In [29]:
df_annot_ready.to_parquet(SINGLE_CELL_READY_PARQUET, index=False)
print("Checkpoint salvo:", SINGLE_CELL_READY_PARQUET)

Checkpoint salvo: /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/cache/single_cell_ready.parquet


## Passo 6 — Agregar por poço (mediana)

A agregação agora usa `df_annot_ready` como entrada.  
Isso substitui o fluxo anterior em que a limpeza vinha **depois** da agregação.


In [30]:
AGG_STRATA = [
    "Metadata_Plate",
    "Metadata_Well",
    "Metadata_Control_Type",
    "Metadata_Treatment",
    "Metadata_Concentration",
    "Metadata_Cell_Type",
]

source_for_agg = (
    df_annot_ready
    if "df_annot_ready" in locals()
    else df_annot_clean if "df_annot_clean" in locals() else df_annot
)

AGG_STRATA = [c for c in AGG_STRATA if c in source_for_agg.columns]

cp_features = infer_cp_features(source_for_agg)

df_agg = (
    source_for_agg.groupby(AGG_STRATA, dropna=False)[cp_features].median().reset_index()
)

print("AGG_STRATA:", AGG_STRATA)
print("df_agg:", df_agg.shape)
display(df_agg.head(3))

AGG_STRATA: ['Metadata_Plate', 'Metadata_Well', 'Metadata_Control_Type', 'Metadata_Treatment', 'Metadata_Concentration', 'Metadata_Cell_Type']
df_agg: (180, 1813)


,Metadata_Plate,Metadata_Well,Metadata_Control_Type,Metadata_Treatment,Metadata_Concentration,Metadata_Cell_Type,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,Cells_AreaShape_Center_Y,Cells_AreaShape_CentralMoment_0_0,Cells_AreaShape_CentralMoment_0_1,Cells_AreaShape_CentralMoment_0_2,Cells_AreaShape_CentralMoment_0_3,Cells_AreaShape_CentralMoment_1_0,Cells_AreaShape_CentralMoment_1_1,Cells_AreaShape_CentralMoment_1_2,Cells_AreaShape_CentralMoment_1_3,Cells_AreaShape_CentralMoment_2_0,Cells_AreaShape_CentralMoment_2_1,Cells_AreaShape_CentralMoment_2_2,Cells_AreaShape_CentralMoment_2_3,Cells_AreaShape_Compactness,Cells_AreaShape_ConvexArea,Cells_AreaShape_Eccentricity,Cells_AreaShape_EquivalentDiameter,Cells_AreaShape_EulerNumber,Cells_AreaShape_Extent,Cells_AreaShape_FormFactor,Cells_AreaShape_HuMoment_0,Cells_AreaShape_HuMoment_1,Cells_AreaShape_HuMoment_2,Cells_AreaShape_HuMoment_3,Cells_AreaShape_HuMoment_4,Cells_AreaShape_HuMoment_5,Cells_AreaShape_HuMoment_6,Cells_AreaShape_InertiaTensorEigenvalues_0,Cells_AreaShape_InertiaTensorEigenvalues_1,Cells_AreaShape_InertiaTensor_0_0,Cells_AreaShape_InertiaTensor_0_1,Cells_AreaShape_InertiaTensor_1_0,Cells_AreaShape_InertiaTensor_1_1,Cells_AreaShape_MajorAxisLength,Cells_AreaShape_MaxFeretDiameter,Cells_AreaShape_MaximumRadius,Cells_AreaShape_MeanRadius,Cells_AreaShape_MedianRadius,Cells_AreaShape_MinFeretDiameter,Cells_AreaShape_MinorAxisLength,Cells_AreaShape_NormalizedMoment_0_0,Cells_AreaShape_NormalizedMoment_0_1,Cells_AreaShape_NormalizedMoment_0_2,Cells_AreaShape_NormalizedMoment_0_3,Cells_AreaShape_NormalizedMoment_1_0,Cells_AreaShape_NormalizedMoment_1_1,Cells_AreaShape_NormalizedMoment_1_2,Cells_AreaShape_NormalizedMoment_1_3,Cells_AreaShape_NormalizedMoment_2_0,Cells_AreaShape_NormalizedMoment_2_1,Cells_AreaShape_NormalizedMoment_2_2,Cells_AreaShape_NormalizedMoment_2_3,Cells_AreaShape_NormalizedMoment_3_0,Cells_AreaShape_NormalizedMoment_3_1,Cells_AreaShape_NormalizedMoment_3_2,Cells_AreaShape_NormalizedMoment_3_3,Cells_AreaShape_Orientation,Cells_AreaShape_Perimeter,Cells_AreaShape_Solidity,Cells_AreaShape_SpatialMoment_0_0,Cells_AreaShape_SpatialMoment_0_1,Cells_AreaShape_SpatialMoment_0_2,Cells_AreaShape_SpatialMoment_0_3,Cells_AreaShape_SpatialMoment_1_0,Cells_AreaShape_SpatialMoment_1_1,Cells_AreaShape_SpatialMoment_1_2,Cells_AreaShape_SpatialMoment_1_3,Cells_AreaShape_SpatialMoment_2_0,Cells_AreaShape_SpatialMoment_2_1,Cells_AreaShape_SpatialMoment_2_2,Cells_AreaShape_SpatialMoment_2_3,Cells_AreaShape_Zernike_0_0,Cells_AreaShape_Zernike_1_1,Cells_AreaShape_Zernike_2_0,Cells_AreaShape_Zernike_2_2,Cells_AreaShape_Zernike_3_1,Cells_AreaShape_Zernike_3_3,Cells_AreaShape_Zernike_4_0,Cells_AreaShape_Zernike_4_2,Cells_AreaShape_Zernike_4_4,Cells_AreaShape_Zernike_5_1,Cells_AreaShape_Zernike_5_3,Cells_AreaShape_Zernike_5_5,Cells_AreaShape_Zernike_6_0,Cells_AreaShape_Zernike_6_2,Cells_AreaShape_Zernike_6_4,...,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_00_256,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_01_256,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_02_256,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_03_256,Nuclei_Texture_SumAverage_AOGFP_10_00_256,Nuclei_Texture_SumAverage_AOGFP_10_01_256,Nuclei_Texture_SumAverage_AOGFP_10_02_256,Nuclei_Texture_SumAverage_AOGFP_10_03_256,Nuclei_Texture_SumAverage_AOGFP_20_00_256,Nuclei_Texture_SumAverage_AOGFP_20_01_256,Nuclei_Texture_SumAverage_AOGFP_20_02_256,Nuclei_Texture_SumAverage_AOGFP_20_03_256,Nuclei_Texture_SumAverage_AOGFP_5_00_256,Nuclei_Texture_SumAverage_AOGFP_5_01_256,Nuclei_Texture_SumAverage_AOGFP_5_02_256,Nuclei_Texture_SumAverage_AOGFP_5_03_256,Nuclei_Texture_SumAverage_AOPI_10_00_256,Nuclei_Texture_SumAverage_AOPI_10_01_256,Nuclei_Texture_SumAverage_AOPI_10_02_256,Nuclei_Texture_SumAverage_AOPI_10_03_25

> **Nota de simplificação**  
> A reanotação de `df_agg` via platemap foi removida.  
> Como a anotação já acontece em `df_annot` e a limpeza é feita antes da agregação, `df_agg` herda os metadados necessários de forma mais consistente.


In [31]:
df_agg.to_parquet(PER_WELL_AGGREGATED_PARQUET, index=False)
print("Checkpoint salvo:", PER_WELL_AGGREGATED_PARQUET)

Checkpoint salvo: /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/cache/per_well_aggregated.parquet


## Passo 7 — Normalizar (plate-wise `mad_robustize`)

In [32]:
# --- Config ----
NORMALIZE_MODE = "negcon"  # ou "all" se quiser normalizar usando todas as amostras
MAD_EPS = 1e-9

from pycytominer.normalize import normalize as pc_normalize
from pycytominer.cyto_utils import infer_cp_features

# --- Enforce negcon-only normalization (sem fallback) ---
if NORMALIZE_MODE == "negcon":
    control_cols = [
        c for c in ("Metadata_Control_Type", "Control_Type") if c in df_agg.columns
    ]
    if not control_cols:
        raise ValueError(
            "NORMALIZE_MODE='negcon' requisitado, mas nenhuma coluna de controle "
            "foi encontrada (esperado: 'Metadata_Control_Type' ou 'Control_Type')."
        )
    col = control_cols[0]
    # normaliza string para comparação robusta
    df_agg[col] = df_agg[col].astype(str).str.strip().str.lower()
    samples_q = f'{col} == "negcon"'

    # garante que existem amostras negcon
    if df_agg.query(samples_q).shape[0] == 0:
        raise ValueError(
            "NORMALIZE_MODE='negcon' requisitado, mas não há linhas com controle negativo "
            f"(condição: {samples_q}). Verifique o plate map/anotação."
        )
else:
    samples_q = None

# --- Seleção de features e metadados ---
cp_features = infer_cp_features(df_agg)
meta_cols = [c for c in df_agg.columns if c.startswith("Metadata_")] + [
    c
    for c in (
        "Control_Type",
        "Treatment",
        "Concentration",
        "Cell_Type",
        "plate_map_name",
    )
    if c in df_agg.columns
]

# --- Normalização ---
kwargs = dict(
    profiles=df_agg,
    features=cp_features,
    method="mad_robustize",
    mad_robustize_epsilon=MAD_EPS,
    meta_features=meta_cols,
)
if samples_q is not None:
    kwargs["samples"] = samples_q

df_norm = pc_normalize(**kwargs)

print("df_norm:", df_norm.shape)
display(df_norm.head(3))

df_norm: (180, 1813)


,Metadata_Plate,Metadata_Well,Metadata_Control_Type,Metadata_Treatment,Metadata_Concentration,Metadata_Cell_Type,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,Cells_AreaShape_Center_Y,Cells_AreaShape_CentralMoment_0_0,Cells_AreaShape_CentralMoment_0_1,Cells_AreaShape_CentralMoment_0_2,Cells_AreaShape_CentralMoment_0_3,Cells_AreaShape_CentralMoment_1_0,Cells_AreaShape_CentralMoment_1_1,Cells_AreaShape_CentralMoment_1_2,Cells_AreaShape_CentralMoment_1_3,Cells_AreaShape_CentralMoment_2_0,Cells_AreaShape_CentralMoment_2_1,Cells_AreaShape_CentralMoment_2_2,Cells_AreaShape_CentralMoment_2_3,Cells_AreaShape_Compactness,Cells_AreaShape_ConvexArea,Cells_AreaShape_Eccentricity,Cells_AreaShape_EquivalentDiameter,Cells_AreaShape_EulerNumber,Cells_AreaShape_Extent,Cells_AreaShape_FormFactor,Cells_AreaShape_HuMoment_0,Cells_AreaShape_HuMoment_1,Cells_AreaShape_HuMoment_2,Cells_AreaShape_HuMoment_3,Cells_AreaShape_HuMoment_4,Cells_AreaShape_HuMoment_5,Cells_AreaShape_HuMoment_6,Cells_AreaShape_InertiaTensorEigenvalues_0,Cells_AreaShape_InertiaTensorEigenvalues_1,Cells_AreaShape_InertiaTensor_0_0,Cells_AreaShape_InertiaTensor_0_1,Cells_AreaShape_InertiaTensor_1_0,Cells_AreaShape_InertiaTensor_1_1,Cells_AreaShape_MajorAxisLength,Cells_AreaShape_MaxFeretDiameter,Cells_AreaShape_MaximumRadius,Cells_AreaShape_MeanRadius,Cells_AreaShape_MedianRadius,Cells_AreaShape_MinFeretDiameter,Cells_AreaShape_MinorAxisLength,Cells_AreaShape_NormalizedMoment_0_0,Cells_AreaShape_NormalizedMoment_0_1,Cells_AreaShape_NormalizedMoment_0_2,Cells_AreaShape_NormalizedMoment_0_3,Cells_AreaShape_NormalizedMoment_1_0,Cells_AreaShape_NormalizedMoment_1_1,Cells_AreaShape_NormalizedMoment_1_2,Cells_AreaShape_NormalizedMoment_1_3,Cells_AreaShape_NormalizedMoment_2_0,Cells_AreaShape_NormalizedMoment_2_1,Cells_AreaShape_NormalizedMoment_2_2,Cells_AreaShape_NormalizedMoment_2_3,Cells_AreaShape_NormalizedMoment_3_0,Cells_AreaShape_NormalizedMoment_3_1,Cells_AreaShape_NormalizedMoment_3_2,Cells_AreaShape_NormalizedMoment_3_3,Cells_AreaShape_Orientation,Cells_AreaShape_Perimeter,Cells_AreaShape_Solidity,Cells_AreaShape_SpatialMoment_0_0,Cells_AreaShape_SpatialMoment_0_1,Cells_AreaShape_SpatialMoment_0_2,Cells_AreaShape_SpatialMoment_0_3,Cells_AreaShape_SpatialMoment_1_0,Cells_AreaShape_SpatialMoment_1_1,Cells_AreaShape_SpatialMoment_1_2,Cells_AreaShape_SpatialMoment_1_3,Cells_AreaShape_SpatialMoment_2_0,Cells_AreaShape_SpatialMoment_2_1,Cells_AreaShape_SpatialMoment_2_2,Cells_AreaShape_SpatialMoment_2_3,Cells_AreaShape_Zernike_0_0,Cells_AreaShape_Zernike_1_1,Cells_AreaShape_Zernike_2_0,Cells_AreaShape_Zernike_2_2,Cells_AreaShape_Zernike_3_1,Cells_AreaShape_Zernike_3_3,Cells_AreaShape_Zernike_4_0,Cells_AreaShape_Zernike_4_2,Cells_AreaShape_Zernike_4_4,Cells_AreaShape_Zernike_5_1,Cells_AreaShape_Zernike_5_3,Cells_AreaShape_Zernike_5_5,Cells_AreaShape_Zernike_6_0,Cells_AreaShape_Zernike_6_2,Cells_AreaShape_Zernike_6_4,...,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_00_256,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_01_256,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_02_256,Nuclei_Texture_InverseDifferenceMoment_AOPI_5_03_256,Nuclei_Texture_SumAverage_AOGFP_10_00_256,Nuclei_Texture_SumAverage_AOGFP_10_01_256,Nuclei_Texture_SumAverage_AOGFP_10_02_256,Nuclei_Texture_SumAverage_AOGFP_10_03_256,Nuclei_Texture_SumAverage_AOGFP_20_00_256,Nuclei_Texture_SumAverage_AOGFP_20_01_256,Nuclei_Texture_SumAverage_AOGFP_20_02_256,Nuclei_Texture_SumAverage_AOGFP_20_03_256,Nuclei_Texture_SumAverage_AOGFP_5_00_256,Nuclei_Texture_SumAverage_AOGFP_5_01_256,Nuclei_Texture_SumAverage_AOGFP_5_02_256,Nuclei_Texture_SumAverage_AOGFP_5_03_256,Nuclei_Texture_SumAverage_AOPI_10_00_256,Nuclei_Texture_SumAverage_AOPI_10_01_256,Nuclei_Texture_SumAverage_AOPI_10_02_256,Nuclei_Texture_SumAverage_AOPI_10_03_25

In [33]:
df_norm.to_parquet(PER_WELL_NORMALIZED_PARQUET, index=False)
print("Checkpoint salvo:", PER_WELL_NORMALIZED_PARQUET)

Checkpoint salvo: /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/cache/per_well_normalized.parquet


## Passo 8 — Feature selection

In [34]:
# --- Imports robustos para diferentes versões do pycytominer ---
try:
    from pycytominer.feature_select import feature_select as pc_feature_select
except Exception:
    # algumas versões expõem em pycytominer.feature_select (módulo) ou diretamente em pycytominer
    try:
        from pycytominer import feature_select as pc_feature_select
    except Exception as e:
        raise ImportError(
            "Não foi possível importar feature_select do pycytominer"
        ) from e

from pycytominer.cyto_utils import infer_cp_features

# --- Parâmetros padrão (defina se ainda não existirem no seu notebook) ---
FS_OPS = [
    "correlation_threshold",
    "variance_threshold",
    "drop_na_columns",
    "drop_outliers",
]
OUTLIER_CUTOFF = 15  # ajuste se quiser mais/menos agressivo
CORR_THRESHOLD = 0.90
NA_CUTOFF = 0.05

# --- Seleção de features (robusto a variações de API) ---
# 1) Detecta features automaticamente
cp_features = infer_cp_features(df_norm)
meta_cols = [c for c in df_norm.columns if c not in cp_features]
features_cols = cp_features


# 2) Chama o feature_select tentando diferentes assinaturas
def run_feature_select_returning_meta(df, features):
    # Tenta com 'return_meta' e argumento 'operation'
    try:
        return pc_feature_select(
            profiles=df,
            features=features,
            operation=FS_OPS,  # algumas versões usam 'operation'
            corr_threshold=CORR_THRESHOLD,
            na_cutoff=NA_CUTOFF,
            outlier_cutoff=OUTLIER_CUTOFF,
            return_meta=True,  # mantém metadados se suportado
        )
    except TypeError:
        # Tenta com 'operations' (plural)
        try:
            return pc_feature_select(
                profiles=df,
                features=features,
                operations=FS_OPS,
                corr_threshold=CORR_THRESHOLD,
                na_cutoff=NA_CUTOFF,
                outlier_cutoff=OUTLIER_CUTOFF,
                return_meta=True,
            )
        except TypeError:
            # Fallback final: sem return_meta -> roda só nas features e reconcatena meta
            try:
                df_fs_only = pc_feature_select(
                    profiles=df,
                    features=features,
                    operation=FS_OPS,
                    corr_threshold=CORR_THRESHOLD,
                    na_cutoff=NA_CUTOFF,
                    outlier_cutoff=OUTLIER_CUTOFF,
                )
            except TypeError:
                # última tentativa com 'operations'
                df_fs_only = pc_feature_select(
                    profiles=df,
                    features=features,
                    operations=FS_OPS,
                    corr_threshold=CORR_THRESHOLD,
                    na_cutoff=NA_CUTOFF,
                    outlier_cutoff=OUTLIER_CUTOFF,
                )
            # Garante que df_fs_only não traga de volta metadados já preservados
            df_fs_only = df_fs_only.drop(
                columns=[c for c in meta_cols if c in df_fs_only.columns],
                errors="ignore",
            ).copy()

            # Reanexa metadados preservando a ordem das linhas
            return pd.concat(
                [
                    df[meta_cols].reset_index(drop=True),
                    df_fs_only.reset_index(drop=True),
                ],
                axis=1,
            )

In [35]:
df_fs = run_feature_select_returning_meta(df_norm, features_cols)
print("df_fs:", df_fs.shape)
display(df_fs.head(3))

df_fs: (180, 351)


,Metadata_Plate,Metadata_Well,Metadata_Control_Type,Metadata_Treatment,Metadata_Concentration,Metadata_Cell_Type,Cells_AreaShape_CentralMoment_0_3,Cells_AreaShape_CentralMoment_1_0,Cells_AreaShape_CentralMoment_1_2,Cells_AreaShape_CentralMoment_2_1,Cells_AreaShape_CentralMoment_2_3,Cells_AreaShape_Compactness,Cells_AreaShape_Eccentricity,Cells_AreaShape_Extent,Cells_AreaShape_FormFactor,Cells_AreaShape_HuMoment_0,Cells_AreaShape_HuMoment_2,Cells_AreaShape_HuMoment_3,Cells_AreaShape_HuMoment_4,Cells_AreaShape_HuMoment_5,Cells_AreaShape_HuMoment_6,Cells_AreaShape_NormalizedMoment_0_2,Cells_AreaShape_NormalizedMoment_0_3,Cells_AreaShape_NormalizedMoment_1_2,Cells_AreaShape_NormalizedMoment_2_0,Cells_AreaShape_NormalizedMoment_2_1,Cells_AreaShape_NormalizedMoment_2_2,Cells_AreaShape_NormalizedMoment_2_3,Cells_AreaShape_NormalizedMoment_3_0,Cells_AreaShape_NormalizedMoment_3_1,Cells_AreaShape_NormalizedMoment_3_2,Cells_AreaShape_NormalizedMoment_3_3,Cells_AreaShape_Perimeter,Cells_AreaShape_Solidity,Cells_AreaShape_Zernike_0_0,Cells_AreaShape_Zernike_1_1,Cells_AreaShape_Zernike_2_0,Cells_AreaShape_Zernike_2_2,Cells_AreaShape_Zernike_3_1,Cells_AreaShape_Zernike_3_3,Cells_AreaShape_Zernike_4_0,Cells_AreaShape_Zernike_4_2,Cells_AreaShape_Zernike_5_1,Cells_AreaShape_Zernike_5_3,Cells_AreaShape_Zernike_6_0,Cells_AreaShape_Zernike_6_2,Cells_AreaShape_Zernike_6_4,Cells_AreaShape_Zernike_6_6,Cells_AreaShape_Zernike_7_1,Cells_AreaShape_Zernike_7_3,Cells_AreaShape_Zernike_7_5,Cells_AreaShape_Zernike_8_0,Cells_AreaShape_Zernike_8_2,Cells_AreaShape_Zernike_8_4,Cells_AreaShape_Zernike_9_1,Cells_AreaShape_Zernike_9_3,Cells_AreaShape_Zernike_9_5,Cells_AreaShape_Zernike_9_7,Cells_AreaShape_Zernike_9_9,Cells_Children_Vesicles_Count,Cells_Granularity_10_AOPI,Cells_Granularity_1_AOGFP,Cells_Granularity_1_AOPI,Cells_Granularity_2_AOPI,Cells_Granularity_3_AOGFP,Cells_Granularity_3_AOPI,Cells_Granularity_5_AOGFP,Cells_Granularity_5_AOPI,Cells_Granularity_6_AOGFP,Cells_Granularity_7_AOGFP,Cells_Granularity_9_AOPI,Cells_Intensity_MassDisplacement_AOGFP,Cells_Intensity_MassDisplacement_AOPI,Cells_Intensity_MinIntensity_AOPI,Cells_Location_MaxIntensity_Y_AOPI,Cells_Mean_Vesicles_AreaShape_BoundingBoxMaximum_X,Cells_Mean_Vesicles_AreaShape_CentralMoment_0_1,Cells_Mean_Vesicles_AreaShape_CentralMoment_0_3,Cells_Mean_Vesicles_AreaShape_CentralMoment_1_0,Cells_Mean_Vesicles_AreaShape_CentralMoment_1_1,Cells_Mean_Vesicles_AreaShape_CentralMoment_1_2,Cells_Mean_Vesicles_AreaShape_CentralMoment_2_1,Cells_Mean_Vesicles_AreaShape_Eccentricity,Cells_Mean_Vesicles_AreaShape_Extent,Cells_Mean_Vesicles_AreaShape_HuMoment_1,Cells_Mean_Vesicles_AreaShape_HuMoment_2,Cells_Mean_Vesicles_AreaShape_InertiaTensor_0_1,Cells_Mean_Vesicles_AreaShape_MedianRadius,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_0_2,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_0_3,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_1_2,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_1_3,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_2_2,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_3_2,Cells_Mean_Vesicles_AreaShape_NormalizedMoment_3_3,Cells_Mean_Vesicles_AreaShape_Orientation,Cells_Mean_Vesicles_AreaShape_Zernike_1_1,Cells_Mean_Vesicles_AreaShape_Zernike_2_0,Cells_Mean_Vesicles_AreaShape_Zernike_2_2,Cells_Mean_Vesicles_AreaShape_Zernike_3_1,...,Nuclei_AreaShape_Zernike_7_5,Nuclei_AreaShape_Zernike_7_7,Nuclei_AreaShape_Zernike_8_0,Nuclei_AreaShape_Zernike_8_2,Nuclei_AreaShape_Zernike_8_4,Nuclei_AreaShape_Zernike_8_6,Nuclei_AreaShape_Zernike_8_8,Nuclei_AreaShape_Zernike_9_1,Nuclei_AreaShape_Zernike_9_3,Nuclei_AreaShape_Zernike_9_5,Nuclei_AreaShape_Zernike_9_7,Nuclei_AreaShape_Zernike_9_9,Nuclei_Correlation_Correlation_AOGFP_AOPI,Nuclei_Correlation_K_AOGFP_AOPI,Nuclei_Correlation_Overlap_AOGFP_AOPI,Nuclei_Correlation_RWC_AOPI_AOGFP,Nuclei_Granularity_10_AOGFP,Nuclei_Granularity_10_AOPI,Nuclei_Granularity_1_AOGFP,Nuclei_Granularity_1_AOPI,Nuclei_Granularity_2_AOGFP,Nuclei_Granularity_2_AOPI,Nuclei_Granular

In [36]:
print("Shape final:", df_fs.shape)
print("Duplicated cols:", df_fs.columns.duplicated().sum())
print("Total NA:", int(df_fs.isna().sum().sum()))
print("Cols with NA:", int((df_fs.isna().sum() > 0).sum()))

Shape final: (180, 351)
Duplicated cols: 0
Total NA: 10
Cols with NA: 10


In [37]:
meta_cols = [c for c in df_fs.columns if c.startswith("Metadata_")]
meta_cols

['Metadata_Plate',
 'Metadata_Well',
 'Metadata_Control_Type',
 'Metadata_Treatment',
 'Metadata_Concentration',
 'Metadata_Cell_Type']

## Passo 9 — Salvar/verificar artefatos finais

In [38]:
def drop_duplicate_and_redundant_cols(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()

    # 0) Diagnóstico
    dups = df2.columns[df2.columns.duplicated()].tolist()
    if dups:
        print(
            "🔁 Colunas com nome duplicado (antes):",
            dups[:30],
            "..." if len(dups) > 30 else "",
        )

    # 1) Remove colunas vindas do platemap com sufixo _pm
    pm_cols = [c for c in df2.columns if c.endswith("_pm")]
    if pm_cols:
        print(f"🧹 Removendo colunas do PM (sufixo _pm): {len(pm_cols)}")
        df2 = df2.drop(columns=pm_cols, errors="ignore")

    # 2) Prefere Metadata_* às versões cruas
    prefer_meta_pairs = [
        ("Treatment", "Metadata_Treatment"),
        ("Concentration", "Metadata_Concentration"),
        ("Control_Type", "Metadata_Control_Type"),
    ]
    to_drop = []
    for raw, meta in prefer_meta_pairs:
        if raw in df2.columns and meta in df2.columns:
            to_drop.append(raw)

    if to_drop:
        print(f"🧹 Removendo versões cruas (preferindo Metadata_*): {to_drop}")
        df2 = df2.drop(columns=to_drop, errors="ignore")

    # 3) Remove nomes exatamente duplicados, mantendo a 1ª ocorrência
    df2 = df2.loc[:, ~df2.columns.duplicated(keep="first")]

    # 4) Checagem final
    still_dup = df2.columns.duplicated().any()
    print("✅ Duplicadas restantes?", still_dup)

    return df2

In [39]:
# nomes claros
df_perwell_raw = df_agg.copy()
df_perwell_norm = df_norm.copy()
df_perwell_norm_fs = df_fs.copy()

# limpa colunas duplicadas/redundantes antes de salvar
df_perwell_raw = drop_duplicate_and_redundant_cols(df_perwell_raw)
df_perwell_norm = drop_duplicate_and_redundant_cols(df_perwell_norm)
df_perwell_norm_fs = drop_duplicate_and_redundant_cols(df_perwell_norm_fs)

# opcional: imputação leve no artefato final
meta_cols = [c for c in df_perwell_norm_fs.columns if c.startswith("Metadata_")]
numeric_feature_cols = df_perwell_norm_fs.select_dtypes(
    include="number"
).columns.tolist()

df_perwell_norm_fs[numeric_feature_cols] = df_perwell_norm_fs[
    numeric_feature_cols
].apply(lambda s: s.fillna(s.median()))

# salva artefatos
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PROFILES_OUT_DIR.mkdir(parents=True, exist_ok=True)

df_perwell_raw.to_parquet(CACHE_DIR / "per_well_aggregated.parquet", index=False)
df_perwell_norm.to_parquet(CACHE_DIR / "per_well_normalized.parquet", index=False)
df_perwell_norm_fs.to_parquet(
    PROFILES_OUT_DIR / "per_well_features_selected.parquet",
    index=False,
)

print("Arquivos salvos em:")
print(" -", (CACHE_DIR / "per_well_aggregated.parquet").resolve())
print(" -", (CACHE_DIR / "per_well_normalized.parquet").resolve())
print(" -", (PROFILES_OUT_DIR / "per_well_features_selected.parquet").resolve())

print(
    "Shapes:",
    "raw",
    df_perwell_raw.shape,
    "| norm",
    df_perwell_norm.shape,
    "| norm+fs",
    df_perwell_norm_fs.shape,
)

✅ Duplicadas restantes? False
✅ Duplicadas restantes? False
✅ Duplicadas restantes? False
Arquivos salvos em:
 - /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/cache/per_well_aggregated.parquet
 - /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/cache/per_well_normalized.parquet
 - /Users/marcelobispojesus/Documents/dev/camila/lcp-nanoplastics-polystyrene-huh7/workspace/profiles/2026_04_Huh7_NPPS_24h/outputs/per_well_features_selected.parquet
Shapes: raw (180, 1813) | norm (180, 1813) | norm+fs (180, 351)


In [40]:
print("Shape:", df_perwell_norm_fs.shape)
print("Duplicated cols:", df_perwell_norm_fs.columns.duplicated().sum())
print("Total NA:", int(df_perwell_norm_fs.isna().sum().sum()))
print(
    "Non-numeric cols:",
    df_perwell_norm_fs.select_dtypes(exclude="number").columns.tolist(),
)

Shape: (180, 351)
Duplicated cols: 0
Total NA: 0
Non-numeric cols: ['Metadata_Plate', 'Metadata_Well', 'Metadata_Control_Type', 'Metadata_Treatment', 'Metadata_Cell_Type']


In [41]:
[c for c in df_perwell_norm_fs.columns if c.startswith("Metadata_")]

['Metadata_Plate',
 'Metadata_Well',
 'Metadata_Control_Type',
 'Metadata_Treatment',
 'Metadata_Concentration',
 'Metadata_Cell_Type']